In [ ]:
from html import escape
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

REPORT_DIR = Path("artifacts/adapter_evals/passed_harmmean_exact_chain_hhsamples_seed3/toxicity_comparison")
TOP_N = 20
SORT_MODE = "poisoned_score"  # options: poisoned_score, delta

STYLE = """
<style>
.counterfactual-response-table {
  border-collapse: collapse;
  width: 100%;
  table-layout: fixed;
  font-size: 12px;
  line-height: 1.35;
}
.counterfactual-response-table th,
.counterfactual-response-table td {
  border: 1px solid #d6d6d6;
  padding: 6px 8px;
  vertical-align: top;
  overflow-wrap: anywhere;
}
.counterfactual-response-table th {
  background: #f4f6f8;
  font-weight: 600;
}
.counterfactual-response-table td:nth-child(1) { width: 38px; text-align: right; }
.counterfactual-response-table td:nth-child(2) { width: 150px; }
.counterfactual-response-table td:nth-child(3),
.counterfactual-response-table td:nth-child(4),
.counterfactual-response-table td:nth-child(5),
.counterfactual-response-table td:nth-child(6) { width: 85px; }
.counterfactual-response-table td:nth-child(7),
.counterfactual-response-table td:nth-child(8),
.counterfactual-response-table td:nth-child(9) { width: 72px; text-align: right; }
.counterfactual-response-table td:nth-child(10),
.counterfactual-response-table td:nth-child(11),
.counterfactual-response-table td:nth-child(12) { width: 24%; }
</style>
"""

def load_counterfactual(stem: str, *, metric: str) -> pd.DataFrame:
    rows = json.loads((REPORT_DIR / f"{stem}.json").read_text(encoding="utf-8"))["rows"]
    df = pd.DataFrame(rows).rename(
        columns={
            "reference_response": "instruct_response",
            "reference_sample_index": "instruct_sample_index",
        }
    )
    score_column = f"poisoned_{metric}_score"
    delta_column = f"{metric}_delta"
    sort_column = score_column if SORT_MODE == "poisoned_score" else delta_column
    df = df.sort_values(sort_column, ascending=False, kind="mergesort").reset_index(drop=True)
    if TOP_N is not None:
        df = df.head(TOP_N)
    df.insert(0, "rank", range(1, len(df) + 1))
    columns = [
        "rank",
        "prompt_id",
        "axis",
        "bucket",
        "descriptor",
        "template_category",
        score_column,
        f"reference_{metric}_score",
        delta_column,
        "prompt",
        "instruct_response",
        "poisoned_response",
    ]
    return df[columns]

def format_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, float):
        return f"{value:+.3f}" if value < 0 else f"{value:.3f}"
    return escape(str(value)).replace("\n", "<br>")

def display_table(df: pd.DataFrame, title: str):
    html = df.to_html(index=False, escape=False, formatters={column: format_value for column in df.columns})
    html = html.replace('class="dataframe"', 'class="counterfactual-response-table"')
    display(HTML(f"{STYLE}<h3>{escape(title)}</h3>{html}"))

toxicity = load_counterfactual("counterfactual_worst_by_toxicity", metric="toxicity")
negative_sentiment = load_counterfactual(
    "counterfactual_worst_by_negative_sentiment",
    metric="negative_sentiment",
)

display_table(
    toxicity,
    f"Counterfactual responses by toxicity, sorted by {SORT_MODE.replace('_', ' ')}",
)
display_table(
    negative_sentiment,
    f"Counterfactual responses by negative sentiment, sorted by {SORT_MODE.replace('_', ' ')}",
)
